In [1]:
#MySQL 연결
import pymysql
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
from google.cloud import bigquery
from dotenv import load_dotenv
import os
import re

#env 파일 읽기
load_dotenv()

True

In [2]:
# 구글 bigquery 연결을 만듭니다.
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../../gcp-credentials.json'

project_id = os.getenv('GCP_PROJECT_ID')
dataset = os.getenv('BQ_DATASET_RC')

big_engine = create_engine(f'bigquery://{project_id}/{dataset}')

client = bigquery.Client(project=project_id)

In [3]:
# [3] raw 데이터 가져오기 (BigQuery)
query = text("""
    select *
    from regional_climate.raw
""")
 
with big_engine.connect() as conn:
    result = conn.execute(query)
    df_raw = pd.DataFrame(result.fetchall(), columns=result.keys())
 
print(f'raw 데이터: {len(df_raw)}행, {len(df_raw.columns)}컬럼')

E:\Data_Project\public-data-analysis\.venv\Lib\site-packages\google\cloud\bigquery\client.py:613: UserWarning: Cannot create BigQuery Storage client, the dependency google-cloud-bigquery-storage is not installed.
  warnings.warn(


raw 데이터: 27668행, 69컬럼


In [4]:
# [3-1] meta 데이터 가져오기 (BigQuery)
query_meta = text("""
    select *
    from regional_climate.meta
""")
 
with big_engine.connect() as conn:
    result = conn.execute(query_meta)
    df_meta = pd.DataFrame(result.fetchall(), columns=result.keys())

In [5]:
#  [4] silver 컬럼 선별 (21개)
# BigQuery에 적재 시 clean_column_name으로 변환된 이름 사용
silver_columns = [
    '지점', '지점명', '일시',
    '평균기온_C', '평균최고기온_C', '평균최저기온_C',
    '최고기온_C', '최저기온_C',
    '평균해면기압_hPa',
    '평균수증기압_hPa',
    '평균이슬점온도_C',
    '평균상대습도',
    '월합강수량_00_24h만_mm',
    '일최다강수량_mm',
    '평균풍속_m_s', '최대풍속_m_s',
    '최다풍향_16방위',
    '평균운량_1_10',
    '합계_일조시간_hr',
    '합계_일사량_MJ_m2',
    '최심적설_cm',
    '평균지면온도_C',
]
 
df = df_raw[silver_columns].copy()
print(f'silver 컬럼 선별 완료: {len(df.columns)}컬럼')

silver 컬럼 선별 완료: 22컬럼


In [6]:
# [5] 숫자 타입 변환
numeric_cols = [c for c in df.columns if c not in ['지점', '지점명', '일시', '최다풍향_16방위']]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
 
df['지점'] = df['지점'].astype(int)

In [7]:
# [6] 일시 파싱 → year, month
df['year'] = df['일시'].str[:4].astype(int)
df['month'] = df['일시'].str[5:7].astype(int)

In [8]:
# [7] 메타 전처리 (transform에서 했던 것과 동일)
df_meta.loc[df_meta['지점명'] == '서청주(예)', '지점주소'] = '충청북도 청주시'
df_meta = df_meta[~df_meta['지점주소'].str.contains('(산지)', na=False, regex=False)]
df_meta = df_meta.drop_duplicates(subset='지점', keep='last')
 
print(f'관측소 메타: {len(df_meta)}개 지점')

관측소 메타: 100개 지점


In [9]:
#[8] 지점주소에서 시도명 추출
df_meta['sd_raw'] = df_meta['지점주소'].str.split().str[0]

In [10]:
# [9] sido_name_alias, sido_name_std 가져오기 (BigQuery common)
query_alias = text(f"""
    select * from `{project_id}.common.sd_name_alias`
""")
query_std = text(f"""
    select * from `{project_id}.common.sd_name_std`
""")

with big_engine.connect() as conn:
    df_alias = pd.DataFrame(conn.execute(query_alias).fetchall(),
                            columns=conn.execute(query_alias).keys())

with big_engine.connect() as conn:
    df_std = pd.DataFrame(conn.execute(query_std).fetchall(),
                          columns=conn.execute(query_std).keys())

print(f'alias: {len(df_alias)}행, std: {len(df_std)}행')

alias: 19행, std: 17행


In [11]:
# [10] 관측소 → sd_code 매핑
# df_meta.sido_raw → sido_name_alias JOIN → sd_code
df_meta = df_meta.merge(
    df_alias,
    left_on='sd_raw',
    right_on='sd_name_alias',
    how='left'
)

# 매핑 확인
missing = df_meta[df_meta['sd_code'].isna()]
if len(missing) > 0:
    print(f'⚠ 시도 매핑 실패 {len(missing)}건:')
    print(missing[['지점', '지점명', 'sd_raw']].to_string())

In [12]:
#  [11] 기후 데이터에 시도 정보 붙이기

# 지점 → sd_code 매핑 추출
station_sido = df_meta[['지점', 'sd_code']].dropna(subset=['sd_code'])
station_sido['sd_code'] = station_sido['sd_code'].astype(int)

# 첫 번째 merge: 기후 데이터에 sd_code 붙이기
df = df.merge(station_sido, on='지점', how='inner')

# 두 번째 merge: sd_code로 표준 시도명 붙이기
df = df.merge(df_std, on='sd_code', how='left')

# 컬럼명을 fact 스키마에 맞추기
df = df.rename(columns={'sd_name': 'sd_name_std'})

print(f'시도 매핑 완료: {len(df)}행, 시도 {df["sd_name_std"].nunique()}개')

시도 매핑 완료: 27032행, 시도 17개


In [13]:
# =============================================================================
# FACT 1: fact_climate_trend (시도 × 연도)
# =============================================================================
# %% [12] fact_climate_trend 생성

# 12개월 미만 연도 제거
month_count = df.groupby(['지점', 'year'])['month'].nunique().reset_index(name='month_cnt')
df_full = df.merge(month_count, on=['지점', 'year'])
df_full = df_full[df_full['month_cnt'] == 12].drop(columns=['month_cnt'])

# Step 1: 관측소별 연간 집계
station_year = df.groupby(['지점', 'sd_code', 'sd_name_std', 'year']).agg(
    avg_temp=('평균기온_C', 'mean'),
    avg_max_temp=('평균최고기온_C', 'mean'),
    avg_min_temp=('평균최저기온_C', 'mean'),
    year_max_temp=('최고기온_C', 'max'),
    year_min_temp=('최저기온_C', 'min'),
    total_precip_mm=('월합강수량_00_24h만_mm', 'sum'),
    avg_humidity_pct=('평균상대습도', 'mean'),
    avg_pressure_hpa=('평균해면기압_hPa', 'mean'),
    total_sunshine_hr=('합계_일조시간_hr', 'sum'),
    avg_ground_temp=('평균지면온도_C', 'mean'),
    avg_max_temp_raw=('평균최고기온_C', 'mean'),
    avg_min_temp_raw=('평균최저기온_C', 'mean'),
).reset_index()

# 관측소별 일교차
station_year['avg_temp_range'] = station_year['avg_max_temp_raw'] - station_year['avg_min_temp_raw']
station_year = station_year.drop(columns=['avg_max_temp_raw', 'avg_min_temp_raw'])

# Step 2: 시도별 평균 (관측소 수 편향 제거)
fact_trend = station_year.groupby(['sd_code', 'sd_name_std', 'year']).agg(
    avg_temp=('avg_temp', 'mean'),
    avg_max_temp=('avg_max_temp', 'mean'),
    avg_min_temp=('avg_min_temp', 'mean'),
    year_max_temp=('year_max_temp', 'max'),
    year_min_temp=('year_min_temp', 'min'),
    total_precip_mm=('total_precip_mm', 'mean'),
    avg_humidity_pct=('avg_humidity_pct', 'mean'),
    avg_pressure_hpa=('avg_pressure_hpa', 'mean'),
    total_sunshine_hr=('total_sunshine_hr', 'mean'),
    avg_ground_temp=('avg_ground_temp', 'mean'),
    avg_temp_range=('avg_temp_range', 'mean'),
).reset_index()

# Step 3: 전년 대비 기온 변화
fact_trend = fact_trend.sort_values(['sd_code', 'year'])
fact_trend['temp_yoy_diff'] = fact_trend.groupby('sd_code')['avg_temp'].diff()

# 소수점 정리
float_cols = fact_trend.select_dtypes(include='float64').columns
fact_trend[float_cols] = fact_trend[float_cols].round(2)
print(f'fact_climate_trend: {len(fact_trend)}행')
print(fact_trend.head())

여기에서 어디 위치에 적용해야 할까?

fact_climate_trend: 440행
   sd_code sd_name_std  year  avg_temp  avg_max_temp  avg_min_temp  \
0       11       서울특별시  2000     12.68         17.12          8.59   
1       11       서울특별시  2001     12.80         17.30          8.92   
2       11       서울특별시  2002     12.86         17.18          9.07   
3       11       서울특별시  2003     12.82         17.01          9.17   
4       11       서울특별시  2004     13.32         17.71          9.52   

   year_max_temp  year_min_temp  total_precip_mm  avg_humidity_pct  \
0           35.1          -12.1           1186.8             63.75   
1           35.3          -18.6           1386.0             60.83   
2           34.8          -12.0           1388.0             62.17   
3           32.2          -15.5           2012.0             64.67   
4           36.2          -16.7           1499.1             62.08   

   avg_pressure_hpa  total_sunshine_hr  avg_ground_temp  avg_temp_range  \
0           1016.07             1506.2            13.62   

In [14]:
# =============================================================================
# FACT 2: fact_regional_compare (시도 × 월, 전체 연도 평년값)
# =============================================================================
# %% [13] fact_regional_compare 생성

# 월별 일교차 계산용 컬럼 추가
df['daily_range'] = df['평균최고기온_C'] - df['평균최저기온_C']

# Step 1: 관측소별 월-연도 값은 이미 df에 있음. 관측소별 월 평년값
station_month = df.groupby(['지점', 'sd_code', 'sd_name_std', 'month']).agg(
    avg_temp=('평균기온_C', 'mean'),
    avg_max_temp=('평균최고기온_C', 'mean'),
    avg_min_temp=('평균최저기온_C', 'mean'),
    avg_daily_range=('daily_range', 'mean'),
    avg_precip_mm=('월합강수량_00_24h만_mm', 'mean'),
    precip_stddev=('월합강수량_00_24h만_mm', 'std'),
    avg_humidity_pct=('평균상대습도', 'mean'),
    avg_wind_speed=('평균풍속_m_s', 'mean'),
    avg_cloud_cover=('평균운량_1_10', 'mean'),
    avg_sunshine_hr=('합계_일조시간_hr', 'mean'),
    avg_solar_mj=('합계_일사량_MJ_m2', 'mean'),
    avg_ground_temp=('평균지면온도_C', 'mean'),
).reset_index()

# Step 2: 시도별 평균
fact_compare = station_month.groupby(['sd_code', 'sd_name_std', 'month']).agg(
    avg_temp=('avg_temp', 'mean'),
    avg_max_temp=('avg_max_temp', 'mean'),
    avg_min_temp=('avg_min_temp', 'mean'),
    avg_daily_range=('avg_daily_range', 'mean'),
    avg_precip_mm=('avg_precip_mm', 'mean'),
    precip_stddev=('precip_stddev', 'mean'),
    avg_humidity_pct=('avg_humidity_pct', 'mean'),
    avg_wind_speed=('avg_wind_speed', 'mean'),
    avg_cloud_cover=('avg_cloud_cover', 'mean'),
    avg_sunshine_hr=('avg_sunshine_hr', 'mean'),
    avg_solar_mj=('avg_solar_mj', 'mean'),
    avg_ground_temp=('avg_ground_temp', 'mean'),
).reset_index()

float_cols = fact_compare.select_dtypes(include='float64').columns
fact_compare[float_cols] = fact_compare[float_cols].round(2)

print(f'fact_regional_compare: {len(fact_compare)}행')
print(fact_compare.head())

fact_regional_compare: 204행
   sd_code sd_name_std  month  avg_temp  avg_max_temp  avg_min_temp  \
0       11       서울특별시      1     -2.01          2.09         -5.64   
1       11       서울특별시      2      0.86          5.43         -3.07   
2       11       서울특별시      3      6.55         11.63          2.22   
3       11       서울특별시      4     12.88         18.20          8.28   
4       11       서울특별시      5     18.48         23.86         13.78   

   avg_daily_range  avg_precip_mm  precip_stddev  avg_humidity_pct  \
0             7.73          19.30          15.42             56.22   
1             8.50          25.97          22.57             54.74   
2             9.40          38.33          33.34             53.88   
3             9.92          73.83          42.61             54.42   
4            10.08          98.60          58.94             59.35   

   avg_wind_speed  avg_cloud_cover  avg_sunshine_hr  avg_solar_mj  \
0            2.34             3.66           175.34    

In [15]:
# =============================================================================
# FACT 3: fact_extreme_weather (시도 × 연도)
# =============================================================================
# %% [14] fact_extreme_weather 생성

# Step 1: 관측소별 연간 극단값
station_extreme = df.groupby(['지점', 'sd_code', 'sd_name_std', 'year']).agg(
    year_max_temp=('최고기온_C', 'max'),
    year_min_temp=('최저기온_C', 'min'),
    max_daily_precip=('일최다강수량_mm', 'max'),
    max_monthly_precip=('월합강수량_00_24h만_mm', 'max'),
    total_precip_mm=('월합강수량_00_24h만_mm', 'sum'),
    max_wind_speed=('최대풍속_m_s', 'max'),
    max_snowfall_cm=('최심적설_cm', 'max'),
).reset_index()

station_extreme['year_temp_range'] = station_extreme['year_max_temp'] - station_extreme['year_min_temp']

# Step 2: 시도별 집계 (극단값은 max, 강수총량은 mean)
fact_extreme = station_extreme.groupby(['sd_code', 'sd_name_std', 'year']).agg(
    year_max_temp=('year_max_temp', 'max'),
    year_min_temp=('year_min_temp', 'min'),
    year_temp_range=('year_temp_range', 'max'),
    max_daily_precip=('max_daily_precip', 'max'),
    max_monthly_precip=('max_monthly_precip', 'max'),
    total_precip_mm=('total_precip_mm', 'mean'),
    max_wind_speed=('max_wind_speed', 'max'),
    max_snowfall_cm=('max_snowfall_cm', 'max'),
).reset_index()

# Step 3: 전년 대비 변화
fact_extreme = fact_extreme.sort_values(['sd_code', 'year'])
fact_extreme['max_daily_precip_yoy_diff'] = fact_extreme.groupby('sd_code')['max_daily_precip'].diff()
fact_extreme['max_temp_yoy_diff'] = fact_extreme.groupby('sd_code')['year_max_temp'].diff()

float_cols = fact_extreme.select_dtypes(include='float64').columns
fact_extreme[float_cols] = fact_extreme[float_cols].round(2)

print(f'fact_extreme_weather: {len(fact_extreme)}행')
print(fact_extreme.head())

fact_extreme_weather: 440행
   sd_code sd_name_std  year  year_max_temp  year_min_temp  year_temp_range  \
0       11       서울특별시  2000           35.1          -12.1             47.2   
1       11       서울특별시  2001           35.3          -18.6             53.9   
2       11       서울특별시  2002           34.8          -12.0             46.8   
3       11       서울특별시  2003           32.2          -15.5             47.7   
4       11       서울특별시  2004           36.2          -16.7             52.9   

   max_daily_precip  max_monthly_precip  total_precip_mm  max_wind_speed  \
0             122.9               599.4           1186.8            12.5   
1             273.4               698.4           1386.0             8.9   
2             178.0               688.0           1388.0            11.7   
3             177.0               684.2           2012.0             9.0   
4             108.5               510.7           1499.1            11.1   

   max_snowfall_cm  max_daily_precip_yoy_

In [16]:
# =============================================================================
# BigQuery 적재
# =============================================================================
# %% [15] BigQuery 적재

def load_to_bq(df, table_name):
    table_id = f'{project_id}.{dataset}.{table_name}'
    job_config = bigquery.LoadJobConfig(
        write_disposition='WRITE_TRUNCATE',
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    print(f'적재 완료: {table_name} → {job.output_rows}행')

load_to_bq(fact_trend, 'fact_climate_trend')
load_to_bq(fact_compare, 'fact_regional_compare')
load_to_bq(fact_extreme, 'fact_extreme_weather')

print('전체 적재 완료')

E:\Data_Project\public-data-analysis\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


적재 완료: fact_climate_trend → 440행


E:\Data_Project\public-data-analysis\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


적재 완료: fact_regional_compare → 204행


E:\Data_Project\public-data-analysis\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


적재 완료: fact_extreme_weather → 440행
전체 적재 완료


In [17]:
fact_trend.describe()

,sd_code,year,avg_temp,avg_max_temp,avg_min_temp,year_max_temp,year_min_temp,total_precip_mm,avg_humidity_pct,avg_pressure_hpa,total_sunshine_hr,avg_ground_temp,avg_temp_range,temp_yoy_diff
count,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,440.000000,423.000000
mean,37.718182,2013.172727,12.973136,18.139818,8.538455,35.362045,-14.367727,1298.253273,66.816591,1016.505455,2132.525273,14.860500,9.601364,-0.479125
std,11.396864,7.836436,2.771143,2.613845,3.137972,3.420818,5.509995,418.446561,5.266627,1.341628,415.426254,2.997549,1.450813,2.644324
min,11.000000,2000.000000,-1.720000,3.180000,-7.300000,15.900000,-29.200000,5.200000,39.000000,1015.010000,291.920000,-0.570000,5.720000,-14.520000
25%,29.000000,2006.000000,12.137500,17.670000,7.370000,34.900000,-18.325000,1084.510000,64.102500,1015.980000,2060.035000,14.235000,8.830000,-0.335000
50%,41.000000,2013.000000,13.295000,18.670000,8.825000,35.900000,-14.200000,1312.415000,67.170000,1016.315000,2207.215000,15.135000,10.035000,0.010000
75%,47.000000,2020.000000,14.425000,19.420000,10.020000,36.925000,-10.675000,1554.675000,70.642500,1016.620000,2345.422500,16.282500,10.712500,0.430000
max,52.000000,2026.000000,17.810000,22.410000,15.010000,41.000000,-0.300000,2328.300000,76.740000,1023.920000,3077.980000,20.020000,12.200000,2.530000
